In [ ]:
import os
import re
import string
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, hamming_loss

TARGET_COLUMNS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE)
    text = re.sub(r"\n", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

def load_data(csv_path="train.csv", test_size=0.2, random_state=42):
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        text_col = "comment_text" if "comment_text" in df.columns else "text"
        df[text_col] = df[text_col].fillna("").astype(str)
    else:
        print(f"Warning: '{csv_path}' not found. Generating sample data for pipeline test...")
        text_col = "comment_text"
        samples = [
            ("Hello, thank you for clarifying this edit on Wikipedia.", [0, 0, 0, 0, 0, 0]),
            ("You are an absolute idiot and should be banned forever!", [1, 0, 1, 0, 1, 0]),
            ("I will find where you live and end you.", [1, 1, 0, 1, 0, 0]),
            ("Why did you revert my changes? Can we discuss on talk page?", [0, 0, 0, 0, 0, 0]),
            ("Get out of here, your group has no place in this community.", [1, 0, 1, 0, 1, 1]),
            ("This article lacks reliable secondary citations.", [0, 0, 0, 0, 0, 0]),
            ("Shut up stupid moron, nobody asked for your opinion.", [1, 0, 1, 0, 1, 0]),
            ("Great work updating the section on machine learning algorithms.", [0, 0, 0, 0, 0, 0]),
        ] * 40
        df = pd.DataFrame({
            "comment_text": [s[0] for s in samples],
            **{col: [s[1][i] for s in samples] for i, col in enumerate(TARGET_COLUMNS)}
        })

    df["cleaned_text"] = df[text_col].apply(clean_text)
    train_df, val_df = train_test_split(df, test_size=test_size, random_state=random_state)
    return train_df, val_df

def evaluate_predictions(model_name, y_true, y_probs, threshold=0.5, save_csv=True):
    y_pred = (y_probs >= threshold).astype(int)

    # Compute per-label ROC-AUC scores
    per_class_auc = {}
    auc_values = []
    for i, col in enumerate(TARGET_COLUMNS):
        try:
            score = roc_auc_score(y_true[:, i], y_probs[:, i])
        except ValueError:
            score = 0.5
        per_class_auc[f"AUC_{col}"] = round(float(score), 4)
        auc_values.append(score)

    mean_auc = round(float(np.mean(auc_values)), 4)
    macro_f1 = round(float(f1_score(y_true, y_pred, average="macro", zero_division=0)), 4)
    micro_f1 = round(float(f1_score(y_true, y_pred, average="micro", zero_division=0)), 4)
    h_loss = round(float(hamming_loss(y_true, y_pred)), 4)

    metrics = {
        "Model": model_name,
        "Mean ROC-AUC": mean_auc,
        "Macro F1": macro_f1,
        "Micro F1": micro_f1,
        "Hamming Loss": h_loss,
        **per_class_auc
    }

    print(f"\n{'='*25} {model_name} Evaluation {'='*25}")
    print(f"Mean ROC-AUC : {mean_auc:<8} | Macro F1 : {macro_f1:<8} | Hamming Loss : {h_loss}")
    print("Per-Label ROC-AUC Breakdown:")
    for col in TARGET_COLUMNS:
        print(f"  - {col:<15}: {per_class_auc[f'AUC_{col}']:.4f}")

    if save_csv:
        log_file = "model_benchmark_log.csv"
        metrics_df = pd.DataFrame([metrics])
        if os.path.exists(log_file):
            existing = pd.read_csv(log_file)
            existing = existing[existing["Model"] != model_name]
            updated = pd.concat([existing, metrics_df], ignore_index=True)
            updated.to_csv(log_file, index=False)
        else:
            metrics_df.to_csv(log_file, index=False)

    return metrics